# 03 — Expérimentations avec MLflow

Ce notebook reprend les modèles étudiés dans le notebook `02_modelisation.ipynb`
afin de suivre et comparer les expérimentations avec MLflow.

L'objectif est d'enregistrer pour chaque run :

- le modèle utilisé ;
- les principaux paramètres ;
- les métriques MAE, RMSE et R² ;
- le modèle entraîné.

In [56]:
import mlflow

print("Version MLflow :", mlflow.__version__)

Version MLflow : 3.15.2


In [57]:
import mlflow.sklearn

In [58]:
experiment = mlflow.set_experiment("prediction_prix_automobile")

print("Nom :", experiment.name)
print("ID :", experiment.experiment_id)

Nom : prediction_prix_automobile
ID : 1


In [59]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [60]:
df = pd.read_csv("../data/interim/vehicles_clean.csv")

df.head()

,YEAR,KM,BODYDOORS,PASSENGERS,MAKE,MODEL,BODYDESCF,DRIVETRAIN,FUEL,TRANSDESCF,COLORF,PRICE
0,2024,14243,4,5,Nissan,Versa,Berline,Traction avant,Essence,Automatique,Gris,18995
1,2021,58800,4,5,Ford,Escape,VUS,4 roues motrices,Essence,Automatique,Noir,18995
2,2019,85525,2,0,Chevrolet,Express Commercial Cutaway,Berline,Propulsion,Essence,Automatique,Blanc,26995
3,2023,60820,4,5,Hyundai,Venue,VUS,Traction avant,Essence,Automatique,Bleu,17995
4,2020,103329,4,5,Volkswagen,Jetta,Berline,Traction avant,Essence,Automatique,Argent,17495


In [61]:
X = df.drop(columns=["PRICE"])
y = df["PRICE"]

print("Dimensions de X :", X.shape)
print("Dimensions de y :", y.shape)

Dimensions de X : (543, 11)
Dimensions de y : (543,)


In [62]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("X_train :", X_train.shape)
print("X_test  :", X_test.shape)
print("y_train :", y_train.shape)
print("y_test  :", y_test.shape)

X_train : (434, 11)
X_test  : (109, 11)
y_train : (434,)
y_test  : (109,)


In [63]:
variables_numeriques = [
    "YEAR",
    "KM",
    "BODYDOORS",
    "PASSENGERS"
]

variables_categorielles = [
    "MAKE",
    "MODEL",
    "BODYDESCF",
    "DRIVETRAIN",
    "FUEL",
    "TRANSDESCF",
    "COLORF"
]

print("Variables numériques :", len(variables_numeriques))
print("Variables catégorielles :", len(variables_categorielles))

Variables numériques : 4
Variables catégorielles : 7


In [64]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            variables_categorielles
        ),
        (
            "num",
            "passthrough",
            variables_numeriques
        )
    ]
)

preprocessor

,transformers,"[('cat', ...), ('num', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,categories,'auto'
,drop,None
,sparse_output,True


In [65]:
model_linear = Pipeline(
    steps=[
        ("preparation", preprocessor),
        ("modele", LinearRegression())
    ]
)

model_linear

,steps,"[('preparation', ...), ('modele', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('cat', ...), ('num', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [66]:
with mlflow.start_run(run_name="linear_regression"):

    # Entraînement
    model_linear.fit(X_train, y_train)

    # Prédictions
    y_pred = model_linear.predict(X_test)

    # Métriques
    mae = mean_absolute_error(y_test, y_pred)
    rmse = mean_squared_error(y_test, y_pred) ** 0.5
    r2 = r2_score(y_test, y_pred)

    # Paramètres enregistrés dans MLflow
    mlflow.log_param("model_name", "LinearRegression")
    mlflow.log_param("test_size", 0.20)
    mlflow.log_param("random_state", 42)

    # Métriques enregistrées dans MLflow
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("r2", r2)

    mlflow.sklearn.log_model(
    sk_model=model_linear,
    name="model"
)

    print("MAE :", mae)
    print("RMSE :", rmse)
    print("R² :", r2)


MAE : 2281.5564111370054
RMSE : 3336.207611950856
R² : 0.754281681416473


In [67]:
from sklearn.linear_model import Ridge

In [68]:
model_ridge = Pipeline(
    steps=[
        ("preparation", preprocessor),
        ("modele", Ridge(alpha=1.0))
    ]
)

In [69]:
with mlflow.start_run(run_name="ridge_alpha_1"):

    # Entraînement
    model_ridge.fit(X_train, y_train)

    # Prédictions
    y_pred = model_ridge.predict(X_test)

    # Métriques
    mae = mean_absolute_error(y_test, y_pred)
    rmse = mean_squared_error(y_test, y_pred) ** 0.5
    r2 = r2_score(y_test, y_pred)

    # Paramètres enregistrés dans MLflow
    mlflow.log_param("model_name", "Ridge")
    mlflow.log_param("alpha", 1.0)
    mlflow.log_param("test_size", 0.20)
    mlflow.log_param("random_state", 42)

    # Métriques enregistrées dans MLflow
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("r2", r2)

    mlflow.sklearn.log_model(
    sk_model=model_ridge,
    name="model"
)

    print("MAE :", mae)
    print("RMSE :", rmse)
    print("R² :", r2)


MAE : 3915.5228117273846
RMSE : 5000.920995192456
R² : 0.4478826635278067


In [70]:
model_ridge = Pipeline(
    steps=[
        ("preparation", preprocessor),
        ("modele", Ridge(alpha=10.0))
    ]
)

In [71]:
with mlflow.start_run(run_name="ridge_alpha_10"):

    # Entraînement
    model_ridge.fit(X_train, y_train)

    # Prédictions
    y_pred = model_ridge.predict(X_test)

    # Métriques
    mae = mean_absolute_error(y_test, y_pred)
    rmse = mean_squared_error(y_test, y_pred) ** 0.5
    r2 = r2_score(y_test, y_pred)

    # Paramètres enregistrés dans MLflow
    mlflow.log_param("model_name", "Ridge")
    mlflow.log_param("alpha", 10.0)
    mlflow.log_param("test_size", 0.20)
    mlflow.log_param("random_state", 42)

    # Métriques enregistrées dans MLflow
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("r2", r2)

    mlflow.sklearn.log_model(
    sk_model=model_ridge,
    name="model"
)

    print("MAE :", mae)
    print("RMSE :", rmse)
    print("R² :", r2)


MAE : 3920.9225936092803
RMSE : 5005.05690032911
R² : 0.4469690521381482


In [72]:
from sklearn.linear_model import Lasso

In [73]:
model_lasso = Pipeline(
    steps=[
        ("preparation", preprocessor),
        ("modele", Lasso(alpha=1.0, max_iter=10000))
    ]
)

In [74]:
with mlflow.start_run(run_name="lasso_alpha_1"):

    # Entraînement
    model_lasso.fit(X_train, y_train)

    # Prédictions
    y_pred = model_lasso.predict(X_test)

    # Métriques
    mae = mean_absolute_error(y_test, y_pred)
    rmse = mean_squared_error(y_test, y_pred) ** 0.5
    r2 = r2_score(y_test, y_pred)

    # Paramètres enregistrés dans MLflow
    mlflow.log_param("model_name", "Lasso")
    mlflow.log_param("alpha", 1.0)
    mlflow.log_param("max_iter", 10000)
    mlflow.log_param("test_size", 0.20)
    mlflow.log_param("random_state", 42)

    # Métriques enregistrées dans MLflow
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("r2", r2)

    mlflow.sklearn.log_model(
    sk_model=model_lasso,
    name="model"
)

    print("MAE :", mae)
    print("RMSE :", rmse)
    print("R² :", r2)

MAE : 1858.3405265748936
RMSE : 2888.8090067173753
R² : 0.8157663077551224


In [75]:
model_lasso = Pipeline(
    steps=[
        ("preparation", preprocessor),
        ("modele", Lasso(alpha=10.0, max_iter=10000))
    ]
)

In [76]:
with mlflow.start_run(run_name="lasso_alpha_10"):

    # Entraînement
    model_lasso.fit(X_train, y_train)

    # Prédictions
    y_pred = model_lasso.predict(X_test)

    # Métriques
    mae = mean_absolute_error(y_test, y_pred)
    rmse = mean_squared_error(y_test, y_pred) ** 0.5
    r2 = r2_score(y_test, y_pred)

    # Paramètres enregistrés dans MLflow
    mlflow.log_param("model_name", "Lasso")
    mlflow.log_param("alpha", 10.0)
    mlflow.log_param("max_iter", 10000)
    mlflow.log_param("test_size", 0.20)
    mlflow.log_param("random_state", 42)

    # Métriques enregistrées dans MLflow
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("r2", r2)

    mlflow.sklearn.log_model(
    sk_model=model_lasso,
    name="model"
)

    print("MAE :", mae)
    print("RMSE :", rmse)
    print("R² :", r2)

MAE : 2113.4504009045727
RMSE : 2894.3321739686116
R² : 0.8150611547090981


In [77]:
from sklearn.ensemble import RandomForestRegressor

In [78]:
model_rf = Pipeline(
    steps=[
        ("preparation", preprocessor),
        (
            "modele",
            RandomForestRegressor(
                n_estimators=100,
                random_state=42
            )
        )
    ]
)

In [79]:
with mlflow.start_run(run_name="random_forest_100"):

    # Entraînement
    model_rf.fit(X_train, y_train)

    # Prédictions
    y_pred = model_rf.predict(X_test)

    # Métriques
    mae = mean_absolute_error(y_test, y_pred)
    rmse = mean_squared_error(y_test, y_pred) ** 0.5
    r2 = r2_score(y_test, y_pred)

    # Paramètres enregistrés dans MLflow
    mlflow.log_param("model_name", "RandomForestRegressor")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("model_random_state", 42)
    mlflow.log_param("test_size", 0.20)
    mlflow.log_param("random_state", 42)

    # Métriques enregistrées dans MLflow
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("r2", r2)

    mlflow.sklearn.log_model(
    sk_model=model_rf,
    name="model"
)

    print("MAE :", mae)
    print("RMSE :", rmse)
    print("R² :", r2)

MAE : 2563.302752293578
RMSE : 3574.4174478847767
R² : 0.7179397154206767


In [80]:
n_estimators = 200

model_rf = Pipeline(
    steps=[
        ("preparation", preprocessor),
        (
            "modele",
            RandomForestRegressor(
                n_estimators=n_estimators,
                random_state=42
            )
        )
    ]
)

In [81]:
with mlflow.start_run(run_name="random_forest_200"):
    # Entraînement
    model_rf.fit(X_train, y_train)

    # Prédictions
    y_pred = model_rf.predict(X_test)

    # Métriques
    mae = mean_absolute_error(y_test, y_pred)
    rmse = mean_squared_error(y_test, y_pred) ** 0.5
    r2 = r2_score(y_test, y_pred)

    # Paramètres enregistrés dans MLflow
    mlflow.log_param("model_name", "RandomForestRegressor")
    mlflow.log_param("n_estimators", 200)
    mlflow.log_param("model_random_state", 42)
    mlflow.log_param("test_size", 0.20)
    mlflow.log_param("random_state", 42)

    # Métriques enregistrées dans MLflow
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("r2", r2)

    mlflow.sklearn.log_model(
    sk_model=model_rf,
    name="model"
)

    print("MAE :", mae)
    print("RMSE :", rmse)
    print("R² :", r2)    

MAE : 2562.110275229358
RMSE : 3552.1827447778146
R² : 0.721437919380693


In [ ]:
import mlflow.sklearn

model_id = "m-50020f47f8f34501a2b596ceece1f8b5"

champion = mlflow.sklearn.load_model(
    f"models:/{model_id}"
)

champion.named_steps["modele"]